> ## SOLUTION / ANSWER KEY &mdash; Lab 10.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-least-privilege.ipynb`](../lab-02-least-privilege.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.2 &mdash; Least Privilege

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Grant only the tools the task actually needs
- Never grant a dangerous (consequential) tool
- Check a tool grant respects least privilege

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; guardrail logic, tool wiring, agent structure, and reading a fixed real message trace &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Safety: guardrails, consolidated](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-10-02"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
The strongest, simplest safety control (deck slides 8&ndash;9): **least privilege**. Give the agent only
the tools its task needs, and **never** a dangerous one &mdash; it cannot misuse a capability it doesn't
have. This limits the **blast radius** when something goes wrong, and it neutralises injection: a hijacked
agent can't do what it **can't**.

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "send_email", "wire_funds", "isolate_host"}
print("consequential tools to withhold unless truly required:", DANGEROUS)

## Your Turn
Complete `grant_tools` (only needed & safe) and `is_least_privilege`.

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "send_email", "wire_funds", "isolate_host"}

def grant_tools(needed, catalog):
    # least privilege: grant only tools that are needed AND not dangerous
    return [t for t in catalog if t in needed and t not in DANGEROUS]

def is_least_privilege(granted, needed):
    # granted must be a subset of needed AND contain no dangerous tool
    return set(granted) <= set(needed) and not any(t in DANGEROUS for t in granted)

try:
    catalog = ["lookup", "compute", "send_email", "summarize"]
    needed  = ["lookup", "compute", "send_email"]   # note: task claims it "needs" send_email
    granted = grant_tools(needed, catalog)
    print('granted:', granted)
    print('least privilege?', is_least_privilege(granted, needed))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a needed safe tool is granted", lambda: "lookup" in grant_tools(["lookup", "compute"], ["lookup", "compute", "summarize"]))
expect_true("a dangerous tool is withheld even if 'needed'", lambda: "send_email" not in grant_tools(["lookup", "send_email"], ["lookup", "send_email"]))
expect_true("tools not needed are excluded", lambda: "summarize" not in grant_tools(["lookup"], ["lookup", "summarize"]))
expect_true("a minimal safe grant is least-privilege", lambda: is_least_privilege(["lookup", "compute"], ["lookup", "compute"]) is True)
expect_true("a grant containing a dangerous tool is NOT least-privilege", lambda: is_least_privilege(["lookup", "wire_funds"], ["lookup", "wire_funds"]) is False)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Grant only what the task needs, never the dangerous tool. The capability an agent doesn't have cannot be misused -- by a bug, a bad reasoning step, or a hijack. Least privilege is the recurring strongest guardrail of the whole course.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>